In [1]:
# general imports
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader
from torch_geometric.nn import global_mean_pool, global_add_pool
from torch_geometric.nn.conv import GraphConv, GINConv, GATv2Conv, TransformerConv
from torch_geometric.utils import from_networkx
from torch_geometric.nn.inits import uniform
import networkx as nx
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import random

C:\Users\eduar\anaconda3\envs\pyg_gnn\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-scatter'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
C:\Users\eduar\anaconda3\envs\pyg_gnn\Lib\site-packages\torch_geometric\__init__.py:4: UserWarning: An issue occurred while importing 'torch-sparse'. Disabling its usage. Stacktrace: [WinError 127] The specified procedure could not be found
  import torch_geometric.typing
C:\Users\eduar\anaconda3\envs\pyg_gnn\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# re-run on cpu only, doesn't need a gpu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [3]:
# base class to introduce init_params, std = 0.01 in authors code repo
class BaseGNN(nn.Module):
    def __init__(self, init_std=0.01):
        super().__init__()
        self.init_std = init_std

    def init_params(self):
        # Generic initialization
        for name, param in self.named_parameters():
            if 'weight' in name and param.dim() > 1:
                nn.init.xavier_normal_(param, gain=self.init_std)
            elif 'bias' in name:
                nn.init.constant_(param, 0)

# note the "add" in the message_passing,
class StandardGraphConv(BaseGNN):
    def __init__(self, in_channels, out_channels, num_layers, hidden_channels, bias=False, **kwargs):
        super().__init__(**kwargs)
        self.convs = nn.ModuleList()
        self.convs.append(GraphConv(in_channels, hidden_channels, aggr='add', bias=bias))
        for _ in range(num_layers - 1):
            self.convs.append(GraphConv(hidden_channels, hidden_channels, aggr='add', bias=bias))

        # global_mean pool over global_add_pool
        self.pool = global_mean_pool
        self.readout = nn.Linear(hidden_channels, out_channels, bias=bias)
        
        # custom init for params needed to be more specific
        self.init_params_graphconv()

    # start W_2 at 0 and otherwise xavier_init as in author repo
    def init_params_graphconv(self):
        for conv in self.convs:
            if hasattr(conv, 'lin_root'): nn.init.xavier_normal_(conv.lin_root.weight, gain=self.init_std)
            if hasattr(conv, 'lin_rel'): nn.init.constant_(conv.lin_rel.weight, 0.0) # W2 starts at 0
        if hasattr(self.readout, 'weight'): nn.init.xavier_normal_(self.readout.weight, gain=self.init_std)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch
        for conv in self.convs:
            x = F.relu(conv(x, edge_index))
        x = self.pool(x, batch)
        return self.readout(x)

In [4]:
# parameters as in paper, we are only generating GNP with p = 0.5
def generate_dataset(num_samples, graph_type, teacher_weights, num_nodes=20, num_features=16):
    dataset = []
    for _ in range(num_samples):
        if graph_type == 'Empty': G = nx.empty_graph(num_nodes)
        elif graph_type == 'Star': G = nx.star_graph(num_nodes - 1)
        elif graph_type == 'Regular': G = nx.random_regular_graph(d=10, n=num_nodes)
        elif graph_type == 'GNP': G = nx.gnp_random_graph(n=num_nodes, p=0.5)
        elif graph_type == 'BA': G = nx.barabasi_albert_graph(n=num_nodes, m=3)

        data = from_networkx(G)
        data.x = torch.randn(num_nodes, num_features)
        graph_sum = data.x.sum(dim=0)
        projection = torch.dot(graph_sum, teacher_weights).item()
        data.y = torch.tensor([1 if projection > 0 else 0], dtype=torch.long)
        dataset.append(data)
    return dataset

In [5]:
import numpy as np

# generic EarlyStopping
class EarlyStopping:
    def __init__(self, patience=100, delta=0, verbose=False):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False

    def __call__(self, val_loss):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
        elif val_loss > self.best_loss + self.delta:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
                if self.verbose:
                    print("Early stopping triggered.")

        return self.early_stop

In [6]:
def train_model(model, train_set, val_set, test_set, epochs=200):
    LR = 0.01
    BATCH_SIZE = 8 
    PATIENCE = 100
    
    optimizer = torch.optim.SGD(model.parameters(), lr=LR, weight_decay = 0.0001)
    criterion = nn.CrossEntropyLoss()
    
    train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

    early_stopper = EarlyStopping(patience=PATIENCE)
    
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            out = model(batch)
            loss = criterion(out, batch.y)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss_sum = 0
        val_total_batches = 0
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(device)
                out = model(batch)
                val_loss_sum += criterion(out, batch.y).item()
                val_total_batches += 1
        
        avg_val_loss = val_loss_sum / val_total_batches

        if early_stopper(avg_val_loss):
            print(f'Early stop triggered at epoch: {epoch}')
            break

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(device)
            out = model(batch)
            pred = out.argmax(dim=1)
            correct += (pred == batch.y).sum().item()
            total += batch.y.size(0)
            
    return correct / total, model

In [7]:
 # constants and config
NUM_NODES = 20
NUM_FEATURES = 128
SAMPLE_SIZE = 4000
GRAPH_TYPES = ['Empty', 'GNP']
NUM_SEEDS = 3
HIDDEN_DIM = 64
OUT_CHANNELS = 2
NUM_LAYERS = 1

results = []

print(f"Starting Experiment with D={NUM_FEATURES}...")

teacher_weights = torch.randn(NUM_FEATURES)
teacher_weights = teacher_weights / torch.norm(teacher_weights)

test_sets = {}
val_sets = {}
print("Generating Test Sets...")

for g_type in GRAPH_TYPES:
    test_sets[g_type] = generate_dataset(500, g_type, teacher_weights, NUM_NODES, NUM_FEATURES)
    val_sets[g_type] = generate_dataset(500, g_type, teacher_weights, NUM_NODES, NUM_FEATURES)

for g_type in GRAPH_TYPES:
    accuracies = []

    print(f"\n--- Graph Type: {g_type} ---")

    for seed in range(NUM_SEEDS):
        current_seed = seed + g_type.__hash__() % 1000 + 42
        torch.manual_seed(current_seed)
        np.random.seed(current_seed)
        random.seed(current_seed)

        train_set = generate_dataset(SAMPLE_SIZE, g_type, teacher_weights, NUM_NODES, NUM_FEATURES)

        model = StandardGraphConv(train_set[0].x.shape[1], OUT_CHANNELS, NUM_LAYERS, HIDDEN_DIM).to(device)

        acc, model = train_model(model, train_set, val_sets[g_type], test_sets[g_type], epochs=1000)
        accuracies.append(acc)

    mean_acc = np.mean(accuracies)

    print(f"Results for {g_type} (N={SAMPLE_SIZE}): Acc={mean_acc:.4f}")

    # Append results for DataFrame
    for i in range(NUM_SEEDS):
        results.append({
            'Number of samples': SAMPLE_SIZE,
            'Accuracy': accuracies[i],
            'Type of graph': g_type
        })

df = pd.DataFrame(results)
print("\n--- Experiment finished. Results DataFrame (df_single_run.csv): ---")
print(df)
df.to_csv("df_single_run.csv", index=False)

Starting Experiment with D=128...
Generating Test Sets...

--- Graph Type: Empty ---
Early stop triggered at epoch: 326
Early stop triggered at epoch: 323
Early stop triggered at epoch: 259
Results for Empty (N=4000): Acc=0.9680

--- Graph Type: GNP ---
Early stop triggered at epoch: 109
Early stop triggered at epoch: 106
Early stop triggered at epoch: 109
Results for GNP (N=4000): Acc=0.9020

--- Experiment finished. Results DataFrame (df_single_run.csv): ---
   Number of samples  Accuracy Type of graph
0               4000     0.960         Empty
1               4000     0.970         Empty
2               4000     0.974         Empty
3               4000     0.908           GNP
4               4000     0.902           GNP
5               4000     0.896           GNP
